In [ ]:
import os
import json
import time
from qgis.core import (
    QgsRasterLayer,
    QgsProject,
    QgsRectangle,
    QgsApplication,
    QgsCoordinateReferenceSystem,
    QgsMapSettings,
    QgsMapRendererParallelJob
)
from qgis.PyQt.QtCore import QSize, QPointF
from qgis.PyQt.QtGui import QImage, QColor, QPainter, QBrush, QPolygonF
from shapely.geometry import shape
from tqdm import tqdm
import pandas as pd
import numpy as np
from shapely import from_wkt

In [ ]:
def initialize_qgis(prefix_path):
    """
    Initialize the QGIS environment.
    """
    os.environ['QT_QPA_PLATFORM'] = 'offscreen'
    QgsApplication.setPrefixPath(prefix_path, True)
    qgs = QgsApplication([], False)
    qgs.initQgis()
    return qgs

In [ ]:
def load_osm_layer():
    """
    Load the OpenStreetMap layer.
    """
    osm_url = "type=xyz&url=https://a.tile.openstreetmap.org/{z}/{x}/{y}.png"
    osm_layer = QgsRasterLayer(osm_url, "OpenStreetMap", "wms")
    if not osm_layer.isValid():
        raise RuntimeError("Failed to load the OpenStreetMap layer!")
    return osm_layer

In [ ]:
def load_geojson(file_path):
    """
    Load a GeoJSON file and return the parsed data.
    """
    try:
        with open(file_path, 'r') as f:
            geojson_data = json.load(f)
        print(f"Successfully loaded GeoJSON file: {file_path}")
        return geojson_data
    except Exception as e:
        raise RuntimeError(f"Failed to load GeoJSON file: {e}")


In [ ]:
def create_output_directory(output_dir):
    """
    Create a directory to save the generated images.
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory created: {output_dir}")


In [ ]:
def render_hexagon_map(hex_polygon, osm_layer, output_path, width=400, height=400):
    """
    Render a hexagonal map and save it as an image.
    """
    min_lon, min_lat, max_lon, max_lat = hex_polygon.bounds
    hex_extent = QgsRectangle(min_lon, min_lat, max_lon, max_lat)

    # Set up map rendering
    map_settings = QgsMapSettings()
    map_settings.setLayers([osm_layer])
    map_settings.setBackgroundColor(QColor("white"))
    map_settings.setOutputSize(QSize(width, height))
    map_settings.setExtent(hex_extent)
    map_settings.setDestinationCrs(QgsCoordinateReferenceSystem("EPSG:4326"))

    # Render the map
    job = QgsMapRendererParallelJob(map_settings)
    job.start()
    job.waitForFinished()

    rendered_image = job.renderedImage()
    if rendered_image.isNull():
        print(f"Failed to render hexagon, image is null!")
        return False

    # Create a transparent mask
    mask_image = QImage(rendered_image.size(), QImage.Format_ARGB32)
    mask_image.fill(QColor(0, 0, 0, 0))
    xform = map_settings.mapToPixel()
    points = [QPointF(xform.transform(x, y).x(), xform.transform(x, y).y()) for x, y in hex_polygon.exterior.coords]
    q_polygon = QPolygonF(points)

    # Draw the hexagonal mask
    painter = QPainter(mask_image)
    painter.setBrush(QBrush(QColor(255, 255, 255, 255)))
    painter.drawPolygon(q_polygon)
    painter.end()

    # Apply the mask
    painter = QPainter(rendered_image)
    painter.setCompositionMode(QPainter.CompositionMode_DestinationIn)
    painter.drawImage(0, 0, mask_image)
    painter.end()

    # Save the image
    rendered_image.save(output_path)
    return True

In [ ]:
def generate_hexagon_images(geodata, osm_layer, output_dir, width=400, height=400, test_mode=False):
    """
    Iterate through GeoJSON data and generate map images for each hexagon.
    """
    for i,hex in tqdm(geodata.iterrows()):
        hex_polygon = shape(hex['geometry'])
        h3_index = hex['h3_index']
        img_filename = f'hex_{i}_{h3_index}_gen.png'
        img_path = os.path.join(output_dir, img_filename)

        if not render_hexagon_map(hex_polygon, osm_layer, img_path, width, height):
            print(f"Failed to render hexagon {i}, skipping.")
            continue

        if test_mode:
            break  # In test mode, only generate one image

In [ ]:
# Configure paths
prefix_path = "c:/Users/Humeruzz/anaconda3/envs/lab_env"
geojson_file_path = 'hex_with_traj.geojson'
output_dir = "./hexagon_images_gen"

# Initialize QGIS
qgs = initialize_qgis(prefix_path)


In [ ]:

try:
    # Load layers and data
    osm_layer = load_osm_layer()
    QgsProject.instance().addMapLayer(osm_layer)

    # geojson_data = load_geojson(geojson_file_path)
    geodata = pd.read_csv('csv_files/hex_data')
    geodata['geometry'] = geodata['geometry'].apply(from_wkt)
    create_output_directory(output_dir)
    # Generate hexagonal map images
    print("Starting to generate hexagonal area images...")
    generate_hexagon_images(geodata, osm_layer, output_dir, test_mode=False)

finally:
    # Close the QGIS application
    qgs.exitQgis()